In [10]:
import sys
from pathlib import Path
import pandas as pd
project_root = Path().resolve().parent
data_path = project_root / "data"

orders = pd.read_csv(data_path/"olist_orders_dataset.csv")
payments = pd.read_csv(data_path/"olist_order_payments_dataset.csv")

delivered_orders = orders[orders["order_status"] == "delivered"].copy()
delivered_orders = delivered_orders[["order_id","order_purchase_timestamp"]]

payments_delivered = payments.merge(
    delivered_orders,
    on="order_id",
    how="inner"
)
payments_delivered["purchase_month"] = pd.to_datetime(
    payments_delivered["order_purchase_timestamp"]
).dt.to_period("M")

monthly_aov = payments_delivered.groupby("purchase_month").agg(
    total_revenue=("payment_value", "sum"),
    total_orders=("order_id", "nunique")
).reset_index()
monthly_aov["aov"] = monthly_aov["total_revenue"] / monthly_aov["total_orders"]
print(monthly_aov)
print(delivered_orders)

   purchase_month  total_revenue  total_orders         aov
0         2016-10       46566.71           265  175.723434
1         2016-12          19.62             1   19.620000
2         2017-01      127545.67           750  170.060893
3         2017-02      271298.65          1653  164.125015
4         2017-03      414369.39          2546  162.753099
5         2017-04      390952.18          2303  169.757785
6         2017-05      567066.73          3546  159.917296
7         2017-06      490225.60          3135  156.371802
8         2017-07      566403.93          3872  146.282007
9         2017-08      646000.61          4193  154.066446
10        2017-09      701169.99          4150  168.956624
11        2017-10      751140.27          4478  167.740123
12        2017-11     1153528.05          7289  158.256009
13        2017-12      843199.17          5513  152.947428
14        2018-01     1078606.86          7069  152.582665
15        2018-02      966510.88          6555  147.4463

In [11]:
delivered_orders["purchase_month"] = pd.to_datetime(
delivered_orders["order_purchase_timestamp"]
).dt.to_period("M")

delivered_orders.groupby("purchase_month")["order_id"].nunique()

payments_delivered.groupby("purchase_month")["order_id"].nunique()

# Compare 
monthly_orders_check = delivered_orders.groupby("purchase_month").agg(
delivered_orders=("order_id", "nunique")
).reset_index()

monthly_payments_check = payments_delivered.groupby("purchase_month").agg(
payment_orders=("order_id", "nunique")
).reset_index()

check = monthly_orders_check.merge(
monthly_payments_check,
on="purchase_month",
how="left"
)

print(check.head(10))


  purchase_month  delivered_orders  payment_orders
0        2016-09                 1             NaN
1        2016-10               265           265.0
2        2016-12                 1             1.0
3        2017-01               750           750.0
4        2017-02              1653          1653.0
5        2017-03              2546          2546.0
6        2017-04              2303          2303.0
7        2017-05              3546          3546.0
8        2017-06              3135          3135.0
9        2017-07              3872          3872.0


2016-09 appears in delivered orders but has no matching payment record in the payments table.
Because monthly AOV requires payment_value, 2016-09 is excluded from the monthly AOV result.
This is a source data consistency issue, not a coding error.